# 🧠 Naive Bayes Classification Worksheet
*Titanic Survival Prediction — Categorical Naive Bayes*

---

Welcome! In this worksheet you will build a **complete Naive Bayes pipeline** on the Titanic dataset using only **categorical/discrete features**. This keeps the math simple and lets you focus on understanding the core algorithm.

> 💡 **How to use this worksheet**
> - Read each markdown cell — it explains **what** and **why**
> - Fill in every `# YOUR CODE HERE` block
> - Hints are provided in each task — use them!
> - Run cells in order from top to bottom

---

## 📋 Pipeline Steps
1. Data Loading & Exploration
2. Feature Selection & Preprocessing
3. Train-Test Split
4. Naive Bayes **from Scratch** (using frequency counts & probabilities)
5. Naive Bayes using **scikit-learn** (`CategoricalNB`)
6. Model Comparison & Evaluation


## 🛳️ Dataset Overview: Titanic

The Titanic dataset contains passenger information. Our goal is to **predict survival** (`Survived = 1` or `Survived = 0`).

### 📊 Features we will use (all discrete/categorical)

| Column | Values | Type |
|--------|--------|------|
| `Pclass` | 1, 2, 3 | Ordinal |
| `Sex` | male, female | Nominal |
| `Embarked` | C, Q, S | Nominal |
| `SibSp` | 0, 1, 2, … | Discrete |
| `Parch` | 0, 1, 2, … | Discrete |

> ✅ We intentionally **skip continuous features** like `Age` and `Fare` here. Simple (categorical) Naive Bayes works purely with **frequency counts** — no Gaussian distributions needed!


## Step 1: Data Loading & Exploration

### 📌 What to do:
- Import `pandas`, `numpy`, `matplotlib.pyplot`, and `seaborn`
- Load the CSV from the URL into a DataFrame called `data`
- Display the **first 5 rows**

> 💡 **Hint:** `pd.read_csv(url)` loads remote CSV files directly into a DataFrame.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# YOUR CODE HERE
# 1. Load the dataset into 'data'


# 2. Display the first 5 rows


### 🔎 Explore the Data

Check data types, missing values, and the distribution of the target column.

> 💡 **Hint:** Use `data.info()`, `data.isnull().sum()`, and `sns.countplot()`.


In [ ]:
# YOUR CODE HERE
# 1. Print data.info() to see column types and null counts


# 2. Print missing values per column


# 3. Plot a count plot of the 'Survived' column


## Step 2: Feature Selection & Preprocessing

We will keep only the **5 categorical/discrete features** listed in the overview, plus the target column `Survived`.

### 📌 What to do:
**A) Select columns:** Keep only `Pclass`, `Sex`, `Embarked`, `SibSp`, `Parch`, and `Survived`.

**B) Handle missing values:**
- `Embarked` has 2 missing values → fill with the **most frequent** value (mode)

**C) Encode text columns:**
- `Sex`: `female` → 0, `male` → 1
- `Embarked`: `C` → 0, `Q` → 1, `S` → 2

> 💡 **Hint for (B):** `data['Embarked'].mode()[0]` gives the most frequent value. Use `.fillna()` to fill missing entries.

> 💡 **Hint for (C):** Use `LabelEncoder` from `sklearn.preprocessing`, or use a dictionary with `.map()` — either works.


In [ ]:
from sklearn.preprocessing import LabelEncoder

# A) Select only the relevant columns
# YOUR CODE HERE
cols = ['Pclass', 'Sex', 'Embarked', 'SibSp', 'Parch', 'Survived']
df = None   # replace None: select these columns from data and drop rows with any NaN


# B) Fill missing values in 'Embarked' with the most frequent value
# YOUR CODE HERE


# C) Encode 'Sex' and 'Embarked' as integers
# YOUR CODE HERE


# Confirm result
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print(df.head())


## 📚 How Simple (Categorical) Naive Bayes Works

### 🔑 Core Idea

For a new passenger with features $X = (x_1, x_2, \dots, x_n)$, we want to find the class $y$ that maximises:

$$P(y \mid X) \propto P(y) \times \prod_{i=1}^{n} P(x_i \mid y)$$

We pick whichever class gives the **higher posterior probability**.

---

### 📐 The Three Quantities

| Quantity | Formula | How to compute |
|----------|---------|----------------|
| **Prior** $P(y)$ | $\frac{\text{count of class } y}{\text{total samples}}$ | Count rows per class |
| **Likelihood** $P(x_i \mid y)$ | $\frac{\text{count}(x_i, y)}{\text{count}(y)}$ | Count feature value within class |
| **Laplace Smoothing** | Add 1 to every count | Prevents zero probabilities |

---

### 🛡️ Why Laplace Smoothing?

If a feature value never appears with a class in training data, the raw probability is **zero**, which kills the entire product. Adding 1 to every count (and adjusting the denominator) avoids this.

$$P(x_i \mid y) = \frac{\text{count}(x_i, y) + 1}{\text{count}(y) + k}$$

where $k$ is the number of unique values of feature $x_i$.

---

### 💡 Example

Suppose we have 3 passengers:

| Pclass | Sex | Survived |
|--------|-----|----------|
| 3 | male | 0 |
| 1 | female | 1 |
| 3 | female | 1 |

For a new passenger: `Pclass=3, Sex=female`:

- $P(Survived=1) = 2/3$
- $P(Pclass=3 \mid Survived=1) = 1/2$
- $P(Sex=female \mid Survived=1) = 2/2 = 1$
- **Posterior(1)** ∝ $2/3 \times 1/2 \times 1 = 0.333$

Do the same for class 0, pick the higher one. ✅


## Step 3: Train-Test Split

Split the cleaned DataFrame into features `X` and target `y`, then into train/test sets.

### 📌 What to do:
- `X` = all columns except `Survived`
- `y` = the `Survived` column
- Use `train_test_split` with `test_size=0.2`, `random_state=42`, `stratify=y`
- Print the sizes of each split

> 💡 **Hint:** `stratify=y` ensures the same survival ratio in both splits.


In [ ]:
from sklearn.model_selection import train_test_split

# YOUR CODE HERE
# 1. Define X and y


# 2. Split into train and test sets


# 3. Print sizes
print(f"Training samples: ")
print(f"Testing samples: ")


## Step 4: Naive Bayes from Scratch 🔧

You will now implement a **Categorical Naive Bayes** classifier using only Python and NumPy.

The class has three methods:

### `fit(X, y)` — Training
Compute and store:
1. **Priors**: `P(class)` for each class
2. **Likelihoods**: `P(feature_value | class)` for every feature, every unique value, every class

Use **Laplace smoothing** (add 1 to counts).

---

### `_log_likelihood(x)` — Score one sample
Given a single sample `x` (one row of feature values), return a dictionary mapping each class to its **log posterior**:

$$\text{log\_posterior}(y) = \log P(y) + \sum_i \log P(x_i \mid y)$$

---

### `predict(X)` — Predict all samples
For each row in `X`, call `_log_likelihood` and return the class with the **highest log posterior**.

---

> 💡 **Storage tip:** Store likelihoods in a nested dictionary:
> `self.likelihoods[feature_name][class][feature_value] = probability`

> 💡 **Log tip:** Use `np.log()`. Add log-priors and log-likelihoods (instead of multiplying probabilities). This avoids underflow.

> 💡 **Unseen values:** If a feature value wasn't seen in training, use the smoothed probability `1 / (class_count + k)` as a fallback.


In [ ]:
import numpy as np

class CategoricalNB_Scratch:
    """
    Categorical Naive Bayes implemented from scratch with Laplace smoothing.
    Suitable for discrete/categorical features (no continuous values).
    """

    def fit(self, X, y):
        """
        Learn priors and likelihoods from training data.

        After fitting, store:
          self.classes       — array of unique class labels
          self.priors        — dict: {class: log P(class)}
          self.likelihoods   — dict: {feature: {class: {value: log P(value|class)}}}
        """
        self.classes = np.unique(y)
        self.priors = {}
        self.likelihoods = {}

        n_total = len(y)

        # --- Step 1: Compute log priors ---
        for cls in self.classes:
            # YOUR CODE HERE
            # Count how many training samples belong to this class
            # Compute P(cls) = count / n_total  and store its LOG
            pass

        # --- Step 2: Compute log likelihoods with Laplace smoothing ---
        for feature in X.columns:
            self.likelihoods[feature] = {}
            k = X[feature].nunique()   # number of unique values for this feature

            for cls in self.classes:
                self.likelihoods[feature][cls] = {}

                # YOUR CODE HERE
                # Get only the rows in X where y == cls
                X_cls = None   # replace None

                n_cls = len(X_cls)   # total samples in this class

                # For each unique value of this feature:
                for val in X[feature].unique():
                    # YOUR CODE HERE
                    # Count how many times 'val' appears for this feature in class cls
                    # Apply Laplace smoothing: (count + 1) / (n_cls + k)
                    # Store the LOG of this probability
                    pass


    def _log_posterior(self, x):
        """
        Compute log posterior for each class for a single sample x.

        x is a pandas Series (one row), so x['Pclass'], x['Sex'], etc.

        Return a dict: {class: log_prior + sum of log_likelihoods}

        Hint: iterate over self.classes, then over self.likelihoods keys (features).
        For each feature, look up self.likelihoods[feature][cls].get(value, fallback).
        """
        scores = {}

        for cls in self.classes:
            # YOUR CODE HERE
            # Start with the log prior
            score = None   # replace None

            for feature in self.likelihoods:
                val = x[feature]
                # YOUR CODE HERE
                # Add the log likelihood for this feature value
                # Use .get(val, fallback) to handle unseen values safely
                pass

            scores[cls] = score

        return scores


    def predict(self, X):
        """
        Predict the class for every row in X.

        Hint: use X.iterrows() to loop over rows,
        call self._log_posterior(row) for each,
        and pick the class with the highest score.

        Return a list of predictions.
        """
        predictions = []

        # YOUR CODE HERE
        for _, row in X.iterrows():
            pass   # replace: compute scores, find argmax class, append to predictions

        return predictions


### ▶️ Train and Predict with Your Scratch Model

Once the class above is complete, run this cell.


In [ ]:
# YOUR CODE HERE
# 1. Create an instance of CategoricalNB_Scratch
# 2. Fit it on X_train, y_train
# 3. Predict on X_test, store result in y_pred_scratch



## Step 5: Naive Bayes with Scikit-learn 📦

Now use scikit-learn's `CategoricalNB`, which is designed exactly for discrete/categorical features.

### 📌 What to do:
- Import `CategoricalNB` from `sklearn.naive_bayes`
- Create the model and fit it on `X_train`, `y_train`
- Predict on `X_test` and store the result in `y_pred_lib`

> 💡 **Hint:** `CategoricalNB` expects non-negative integer inputs — your encoded features already satisfy this.

> 💡 **Laplace smoothing** is controlled by the `alpha` parameter (default = 1.0), which matches what you implemented from scratch.


In [ ]:
from sklearn.naive_bayes import CategoricalNB

# YOUR CODE HERE
# 1. Create a CategoricalNB model (use default alpha=1.0)


# 2. Fit on training data


# 3. Predict on test data, store in y_pred_lib



## Step 6: Model Comparison & Evaluation 📊

Evaluate both models using standard classification metrics.

### 📌 Metrics to compute:

| Metric | Meaning |
|--------|---------|
| **Accuracy** | % of total predictions that are correct |
| **Precision** | Of predicted survivors, how many actually survived |
| **Recall** | Of actual survivors, how many did we correctly identify |
| **F1-score** | Balanced measure of precision and recall |
| **Confusion Matrix** | TP / FP / TN / FN breakdown |

### 📌 What to do:
- Complete the `evaluate_model` function
- Call it once for the scratch model and once for the sklearn model
- Compare the results

> 💡 **Hint:** Use `accuracy_score`, `classification_report`, `confusion_matrix` from `sklearn.metrics`.  
> For the heatmap: `sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])`


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate_model(name, y_true, y_pred):
    """
    Print accuracy + classification report and show a confusion matrix heatmap.
    """
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")

    # YOUR CODE HERE
    # 1. Print accuracy_score

    # 2. Print classification_report

    # 3. Compute confusion matrix and plot as heatmap
    pass


In [ ]:
# Evaluate scratch implementation
evaluate_model("Categorical NB — From Scratch", y_test, y_pred_scratch)


In [ ]:
# Evaluate scikit-learn implementation
evaluate_model("Categorical NB — Scikit-learn", y_test, y_pred_lib)


## 🔍 Manual Probability Check (Optional but Highly Recommended!)

Pick **one specific passenger** from the test set and manually trace through the Naive Bayes calculation yourself to verify your model's prediction.

### 📌 What to do:
- Choose a row from `X_test` (e.g., the first row)
- Print its feature values
- Manually compute `P(Survived=1 | features)` and `P(Survived=0 | features)` step by step
- Check that your scratch model predicts the same class

> 💡 **Hint:** Use `X_test.iloc[0]` to get the first test row and `scratch_model._log_posterior(row)` to see the scores.


In [ ]:
# YOUR CODE HERE
# 1. Pick the first row of X_test


# 2. Print its feature values


# 3. Call _log_posterior on it and print the class scores


# 4. Print what the model predicted vs the actual label


## 🤔 Reflection Questions

Think about and answer the following questions (as comments or in a markdown cell):

1. Your scratch model and the sklearn model should give very similar (or identical) accuracy. If they differ slightly, what might cause the difference?
2. What does the confusion matrix tell you about which class the model predicts better?
3. Why do we use **log probabilities** instead of multiplying raw probabilities directly?
4. What is **Laplace smoothing** and what problem does it solve? What happens if you set `alpha=0` in `CategoricalNB`?
5. We dropped `Age` and `Fare` in this worksheet. How would you include them in a Naive Bayes model? Which variant would you use?
6. *(Bonus)* The Naive Bayes assumption is that all features are **independent**. Do you think `Pclass` and `Fare` are truly independent? What effect might this have on the model?


In [ ]:
# Write your answers as comments here:

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:

# Q6 (Bonus):
